# 01 - Resume NER: data collection, cleaning, EDA

**Goal:** build a clean, split, labelled dataset for extracting entities (Name, College, Degree, Skills,
Designation, Companies, ...) from resumes.

Runtime: CPU is enough. Run the cells top to bottom.

Plan for this notebook: collect -> convert to BIO tags -> clean -> de-duplicate -> EDA -> split by resume -> save.

In [ ]:
import os, sys, json, glob
if not os.path.exists("InterviewAI"):
    !git clone -q https://github.com/Ishangupta-cyber/InterviewAI.git
%cd /content/InterviewAI
!git pull -q
sys.path.insert(0, "ml/src/resume_ner")
import data

OUT = "/content/drive/MyDrive/interview_ai_ml"   # splits, models and reports live here, so Colab resets cannot erase them
try:
    from google.colab import drive  # type: ignore  (this module only exists on Colab)
    drive.mount("/content/drive")
except Exception as e:   # Drive mount may be unavailable (e.g. running through VS Code)
    print("Drive not mounted (" + type(e).__name__ + "). Saving to /content instead; copy files out before the runtime ends.")
    OUT = "/content/interview_ai_ml"
os.makedirs(OUT, exist_ok=True)
print("outputs ->", OUT)

!pip -q install seqeval pandas matplotlib

## 1. Collect

Source: the **Resume Entities for NER** dataset (DataTurks, hand-labelled resumes from indeed.com, 10 labels).
This notebook downloads the public GitHub copy (200 resumes), so no account or `kaggle.json` is needed.
The Kaggle version has 220 resumes: if you want it, download it from
kaggle.com/datasets/dataturks/resume-entities-for-ner and put the file in `ml/data/raw/`.
Check the dataset licence and cite it in the report.

In [ ]:
import urllib.request
os.makedirs("ml/data/raw", exist_ok=True)
URL = "https://raw.githubusercontent.com/DataTurks-Engg/Entity-Recognition-In-Resumes-SpaCy/master/traindata.json"
urllib.request.urlretrieve(URL, "ml/data/raw/traindata.json")
print(os.listdir("ml/data/raw"))

In [ ]:
path = glob.glob("ml/data/raw/*.json*")[0]
records = data.load_jsonl(path)
print(path, "->", len(records), "resumes")
print("keys:", list(records[0].keys()))
print("first annotation:", records[0]["annotation"][0])

### 1b. (Optional) a bigger dataset

`yashpwr/resume-ner-training-data` on Hugging Face (and Kaggle *Resume NER Training Dataset*) combine several sources
(about 6k resumes). **I have not verified its column format.** Print it, and if it already has `tokens` + `ner_tags`
you can use those directly instead of step 2. Keep the same label names so results stay comparable, and
record which dataset you used in the report.

In [ ]:
from datasets import load_dataset
ds = load_dataset("yashpwr/resume-ner-training-data")
print(ds)
print(ds["train"][0])

## 2. Convert to BIO tokens, clean, de-duplicate

BIO tagging: `B-SKILLS` starts a skills span, `I-SKILLS` continues it, `O` is outside any entity.
Cleaning is token-level (unicode normalisation, control characters), because the labels are character offsets
into the original text, and editing the text first would shift them.

In [ ]:
examples = data.build_examples(records)
print("resumes with tokens:", len(examples))
examples = data.dedupe(examples)
print("after removing exact duplicates:", len(examples))

labelled = sum(t != "O" for ex in examples for t in ex["tags"])
total = sum(len(ex["tags"]) for ex in examples)
print(f"tokens: {total:,}, inside an entity: {labelled:,} ({labelled/total:.1%})")
print(list(zip(examples[0]["tokens"][:25], examples[0]["tags"][:25])))

## 3. EDA

Questions to answer (and write up): Which labels are rare? How long are resumes? Is any label missing in many resumes?
Rare labels will have low F1, and the report should say so.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from collections import Counter

counts = data.entity_counts(examples)
lengths = [len(ex["tokens"]) for ex in examples]
rows = []
for lab in counts:
    spans = []
    for ex in examples:
        n = 0
        for t in ex["tags"] + ["O"]:
            if t == "B-" + lab:
                if n: spans.append(n)
                n = 1
            elif t == "I-" + lab: n += 1
            else:
                if n: spans.append(n)
                n = 0
    present = sum(any(t.endswith("-" + lab) for t in ex["tags"]) for ex in examples)
    rows.append({"label": lab, "spans": counts[lab], "resumes_with_label": present,
                 "pct_resumes": round(100 * present / len(examples), 1),
                 "avg_words_per_span": round(sum(spans) / max(len(spans), 1), 1)})
eda = pd.DataFrame(rows).sort_values("spans", ascending=False)
display(eda)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
eda.plot.bar(x="label", y="spans", ax=ax[0], legend=False, title="Entity spans per label")
ax[1].hist(lengths, bins=30); ax[1].set_title("Resume length (words)"); ax[1].set_xlabel("words")
plt.tight_layout(); plt.savefig(f"{OUT}/eda.png", dpi=150); plt.show()
print("resume length: min", min(lengths), "median", sorted(lengths)[len(lengths)//2], "max", max(lengths))

**Write down 3 findings from the table and charts** (for example: "Skills spans are very long because the dataset labels a whole
skills paragraph as one span", "Graduation Year appears in only X% of resumes"). Your teacher will ask what the EDA told you.

## 4. Split by resume and save

Split **before** chunking, so no resume appears in two splits (that would leak test data into training).
The test set is used once, in notebook 03.

In [ ]:
train, val, test = data.split(examples, seed=42, ratios=(0.8, 0.1, 0.1))
labels = data.label_list(examples)
print("train/val/test resumes:", len(train), len(val), len(test))
print("labels:", labels)

for name, part in [("train", train), ("val", val), ("test", test)]:
    data.save_jsonl(part, f"{OUT}/{name}_resumes.jsonl")
    print(name, dict(data.entity_counts(part)))
json.dump(labels, open(f"{OUT}/labels.json", "w"))
print("saved to", OUT)